In [1]:

# 1. Imports & Setup

!pip install -q lightgbm scikit-learn pandas numpy

import pandas as pd
import numpy as np
import re
from collections import Counter
import math
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.naive_bayes import ComplementNB
import joblib
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')
def seed_everything(seed=42):
    np.random.seed(seed)
seed_everything(42)

In [2]:

# 2. Innovative Feature Engineering 

import math
from collections import Counter
import re

def shannon_entropy(text):
    """Calculates character entropy. High entropy indicates obfuscation/base64."""
    if not text: return 0
    p, lns = Counter(text), float(len(text))
    return -sum(count/lns * math.log2(count/lns) for count in p.values())

def lexical_diversity(text):
    """Ratio of unique words to total words. Low ratio often means repetition attacks."""
    words = text.lower().split()
    if not words: return 0
    return len(set(words)) / len(words)

def extract_nlp_features(df):
    """Generates structural and statistical metadata from the text."""
    print("Extracting statistical NLP features...")
    features = pd.DataFrame()
    
    # Base features
    features['length'] = df['Prompt'].astype(str).apply(len)
    features['word_count'] = df['Prompt'].astype(str).apply(lambda x: len(x.split()))
    features['entropy'] = df['Prompt'].astype(str).apply(shannon_entropy)
    features['lexical_diversity'] = df['Prompt'].astype(str).apply(lexical_diversity)
    
    # Structural Heuristics
    features['special_char_ratio'] = df['Prompt'].astype(str).apply(lambda x: len(re.findall(r'[^a-zA-Z0-9\s]', x)) / (len(x) + 1))
    features['uppercase_ratio'] = df['Prompt'].astype(str).apply(lambda x: len(re.findall(r'[A-Z]', x)) / (len(x) + 1))
    
    # NEW: Adversarial Keyword Density
    jailbreak_words = ['ignore', 'bypass', 'system', 'instruction', 'prompt', 'developer mode', 'act as', 'override']
    features['keyword_count'] = df['Prompt'].astype(str).str.lower().apply(
        lambda x: sum(1 for word in jailbreak_words if word in x)
    )
    
    return features

In [3]:

# 3. ULTRA-RESOLUTION Vectorization 

def get_advanced_tfidf(train_texts, test_texts):
    print("Generating Ultra-Resolution TF-IDF matrices...")
    
    # Expanded to 25,000 words
    word_vect = TfidfVectorizer(
        max_features=25000, 
        stop_words='english', 
        ngram_range=(1, 3),
        sublinear_tf=True 
    )
    
    # Expanded to 35,000 char-ngrams, catching micro-patterns (2 to 6 chars)
    char_vect = TfidfVectorizer(
        max_features=35000, 
        analyzer='char_wb', 
        ngram_range=(2, 6),
        sublinear_tf=True
    )
    
    train_word = word_vect.fit_transform(train_texts)
    train_char = char_vect.fit_transform(train_texts)
    
    test_word = word_vect.transform(test_texts)
    test_char = char_vect.transform(test_texts)
    
    X_train_tfidf = hstack([train_word, train_char]).tocsr() 
    X_test_tfidf = hstack([test_word, test_char]).tocsr()
    
    joblib.dump(word_vect, 'word_vectorizer.pkl')
    joblib.dump(char_vect, 'char_vectorizer.pkl')
    
    return X_train_tfidf, X_test_tfidf



In [4]:

# 4. Data Processing Pipeline (CLEANED)

# Loading the data
train_df = pd.read_csv('/kaggle/input/competitions/data-sprint-prompt-based-classififcation/merged_train_70.csv')
test_df = pd.read_csv('/kaggle/input/competitions/data-sprint-prompt-based-classififcation/merged_test_30.csv')

# Droping rows where the target label is missing
train_df = train_df.dropna(subset=['isMalicious']).reset_index(drop=True)

# Handleing missing text
train_df['Prompt'] = train_df['Prompt'].fillna("").astype(str)
test_df['Prompt'] = test_df['Prompt'].fillna("").astype(str)

print(f"Training on {len(train_df)} rows after cleaning...")

# 1. EXTRACTING NLP FEATURES FIRST
print("--- Processing Data Features ---")
train_nlp_features = extract_nlp_features(train_df)
test_nlp_features = extract_nlp_features(test_df)

# 2. GENERATING TF-IDF MATRICES SECOND
print("--- Processing Text Embeddings (TF-IDF) ---")
X_train_tfidf, X_test_tfidf = get_advanced_tfidf(train_df['Prompt'], test_df['Prompt'])

# 3. COMBINING THEM SAFELY USING HSTACK
# We convert the pandas features to .values so they stack cleanly with the sparse TF-IDF matrices
X_train = hstack([train_nlp_features.values, X_train_tfidf]).tocsr()
X_test = hstack([test_nlp_features.values, X_test_tfidf]).tocsr()

# 4. SETING THE TARGET VARIABLE
y_train = train_df['isMalicious'] 

print("✅ Data Pipeline Complete! Ready for Model Training.")

Training on 27531 rows after cleaning...
--- Processing Data Features ---
Extracting statistical NLP features...
Extracting statistical NLP features...
--- Processing Text Embeddings (TF-IDF) ---
Generating Ultra-Resolution TF-IDF matrices...
✅ Data Pipeline Complete! Ready for Model Training.


In [5]:

# 5. 3-Way Ensemble & Mathematical Weight Optimization

print("\n--- Training 3-Way Ensemble ---")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(train_df))
oof_lr = np.zeros(len(train_df))
oof_cnb = np.zeros(len(train_df))

lgb_params = {
    'objective': 'binary', 'metric': 'binary_logloss', 
    'learning_rate': 0.05, 'num_leaves': 127, 'max_depth': -1,             
    'feature_fraction': 0.6, 'verbose': -1, 'random_state': 42, 'n_jobs': -1                
}
best_iteration_list = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train[val_idx], y_train.iloc[val_idx]
    
    # 1. LightGBM (Deep Trees)
    lgb_model = lgb.train(
        lgb_params, lgb.Dataset(X_tr, label=y_tr),
        valid_sets=[lgb.Dataset(X_va, label=y_va)],
        num_boost_round=1500, callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    oof_lgb[val_idx] = lgb_model.predict(X_va, num_iteration=lgb_model.best_iteration)
    best_iteration_list.append(lgb_model.best_iteration)
    
    # 2. Logistic Regression (Linear Sparse Master)
    lr_model = LogisticRegression(C=5.0, max_iter=1000, solver='liblinear', random_state=42).fit(X_tr, y_tr)
    oof_lr[val_idx] = lr_model.predict_proba(X_va)[:, 1]
    
    # 3. Complement Naive Bayes (Imbalanced Text Specialist)
    cnb_model = ComplementNB(alpha=0.5).fit(X_tr, y_tr)
    oof_cnb[val_idx] = cnb_model.predict_proba(X_va)[:, 1]
    
    print(f"Fold {fold+1} Finished.")

# --- DYNAMIC WEIGHT & THRESHOLD SEARCH ---
print("\nSearching for mathematically optimal ensemble weights...")
best_f1 = 0.0
best_thresh = 0.5
best_w = (0.33, 0.33, 0.34) # Default fallback

# Brute force search through possible weight combinations
for w_lgb in np.arange(0.0, 1.01, 0.1):
    for w_lr in np.arange(0.0, 1.01 - w_lgb, 0.1):
        w_cnb = 1.0 - w_lgb - w_lr
        
        # Create this specific blend
        blend = (w_lgb * oof_lgb) + (w_lr * oof_lr) + (w_cnb * oof_cnb)
        
        # Scan thresholds for this specific blend
        for thresh in np.arange(0.30, 0.71, 0.02):
            preds = (blend >= thresh).astype(int)
            score = f1_score(y_train, preds)
            
            if score > best_f1:
                best_f1 = score
                best_thresh = thresh
                best_w = (w_lgb, w_lr, w_cnb)

print(f"\n✅ Optimal Weights Found -> LGBM: {best_w[0]:.2f} | LR: {best_w[1]:.2f} | CNB: {best_w[2]:.2f}")

# --- CALCULATE FINAL METRICS ---
final_oof_blend = (best_w[0] * oof_lgb) + (best_w[1] * oof_lr) + (best_w[2] * oof_cnb)
final_binary_preds = (final_oof_blend >= best_thresh).astype(int)

print(f"\n======================================")
print(f"🏆 FINAL ENSEMBLE METRICS 🏆")
print(f"======================================")
print(f"Optimal Threshold : {best_thresh:.3f}")
print(f"F1 Score          : {best_f1:.5f}" )
print(f"Accuracy          : {accuracy_score(y_train, final_binary_preds):.5f}")
print(f"Precision         : {precision_score(y_train, final_binary_preds):.5f}")
print(f"Recall            : {recall_score(y_train, final_binary_preds):.5f}")
print(f"ROC-AUC           : {roc_auc_score(y_train, final_oof_blend):.5f}")
print(f"\n--- Mandatory Confusion Matrix ---")
print(confusion_matrix(y_train, final_binary_preds))
print(f"======================================\n")

# --- TRAIN EXPORTABLE MODELS ---
print("Training final models on 100% of data for submission...")
final_lgb = lgb.train(lgb_params, lgb.Dataset(X_train, label=y_train), num_boost_round=int(np.mean(best_iteration_list)))
final_lr = LogisticRegression(C=5.0, max_iter=1000, solver='liblinear', random_state=42).fit(X_train, y_train)
final_cnb = ComplementNB(alpha=0.5).fit(X_train, y_train)

final_lgb.save_model('lightgbm_final_model.txt')
joblib.dump(final_lr, 'logistic_regression_final.pkl')
joblib.dump(final_cnb, 'complement_nb_final.pkl')
print("✅ All Final Models saved to disk.")


--- Training 3-Way Ensemble ---
Fold 1 Finished.
Fold 2 Finished.
Fold 3 Finished.
Fold 4 Finished.
Fold 5 Finished.

Searching for mathematically optimal ensemble weights...

✅ Optimal Weights Found -> LGBM: 0.70 | LR: 0.20 | CNB: 0.10

🏆 FINAL ENSEMBLE METRICS 🏆
Optimal Threshold : 0.300
F1 Score          : 0.96585
Accuracy          : 0.96644
Precision         : 0.98322
Recall            : 0.94908
ROC-AUC           : 0.99086

--- Mandatory Confusion Matrix ---
[[13542   223]
 [  701 13065]]

Training final models on 100% of data for submission...
✅ All Final Models saved to disk.


In [8]:

# 6. Final Submission Generation 

print("\nGenerating blended test predictions...")
test_preds_lgb = final_lgb.predict(X_test)
test_preds_lr = final_lr.predict_proba(X_test)[:, 1]
test_preds_cnb = final_cnb.predict_proba(X_test)[:, 1]

# Apply the mathematically optimal weights found in Step 5
final_test_probs = (best_w[0] * test_preds_lgb) + (best_w[1] * test_preds_lr) + (best_w[2] * test_preds_cnb)

# Apply the tuned threshold
final_labels = (final_test_probs >= best_thresh).astype(int)

submission = pd.DataFrame({
    'id': test_df['id'].fillna(0).astype(int),
    'label': final_labels            
})

submission.to_csv('submission.csv', index=False)
print("✅ Blended Submission saved successfully as 'submission.csv'!")


Generating blended test predictions...
✅ Blended Submission saved successfully as 'submission.csv'!
